In [ ]:
import pandas as pd
import geopandas as gpd
import requests

In [ ]:
url = "https://maps2.columbus.gov/arcgis/rest/services/Schemas/RecreationParks/MapServer/1/?f=json"

In [ ]:
r = requests.get(url)
serviceInfo = r.json()

In [ ]:
fields = {}
for field in serviceInfo["fields"]:
    fields[field["name"]] = field

In [ ]:
url = "https://maps2.columbus.gov/arcgis/rest/services/Schemas/RecreationParks/MapServer/1/query"
params = {
    "where":"1=1",
    "returnCountOnly":"true",
    "f":"json"
}

In [ ]:
r = requests.get(url, params)

In [ ]:
recordCount = r.json()["count"]

In [ ]:
recordCount

In [ ]:
requestCount = 2000

In [ ]:
recordIndex = 0
url = "https://maps2.columbus.gov/arcgis/rest/services/Schemas/RecreationParks/MapServer/1/query"
while recordIndex < recordCount:
    params = {
        "outFields":"*",
        "where":"1=1",
        "f":"geojson",
        "resultOffset":recordIndex,
        "resultRecordCount":requestCount
    }
    r = requests.get(url, params)
    json = r.json()
    temp = gpd.GeoDataFrame.from_features(json)
    if(recordIndex == 0):
        treesRaw = temp.copy()
    else:
        treesRaw = pd.concat([treesRaw, temp], axis="index")
    recordIndex = recordIndex + requestCount + 1

In [ ]:
trees = treesRaw.copy()

In [ ]:
trees.crs = "epsg:4326"

In [ ]:
trees.columns

In [ ]:
valueMap = {}
for item in fields["CONDITION1"]["domain"]["codedValues"]:
    valueMap[item["code"]] = item["name"]
trees["Condition"] = trees["CONDITION1"].map(valueMap)

In [ ]:
trees = trees.loc[trees["Condition"].isin(["Good","Fair","Poor","Excellent"])].copy()

In [ ]:
valueMap = {}
for item in fields["SP_CODE"]["domain"]["codedValues"]:
    valueMap[item["code"]] = item["name"]
trees["Species"] = trees["SP_CODE"].map(valueMap)

In [ ]:
trees = trees.rename(columns={"HEIGHT":"Height","DIAM_BREAST_HEIGHT":"Trunk diameter","ADDRESS":"Address"})

In [ ]:
trees = trees.filter(items=["OBJECTID","Species","Condition","Height","Address","geometry"], axis="columns")

In [ ]:
trees = trees.loc[trees["Species"].str.lower().str.contains("apple") == True].copy()

In [ ]:
m = trees.explore(column="Condition", cmap="Set1", marker_kwds={"radius":4})

In [ ]:
m.save("map.html")

In [ ]:
trees.to_file("trees.gpkg", layer="trees")